# DINO v3: Aprendizaje Auto-Supervisado para Vision

DINOv2 (y DINOv3) son modelos que aprenden features visuales sin necesidad de etiquetas. Aprenden mirando millones de imagenes y encontrando patrones por si mismos.

**Para que sirve:**
- Extraer features (embeddings) de imagenes
- Encontrar similitudes entre imagenes o regiones
- Servir como base (backbone) para otros modelos
- Clustering y busqueda de imagenes similares

La ventaja: no necesita datos etiquetados, aprende conceptos generales de manera automatica.

## Configuracion e Imports

In [ ]:
import torch
import transformers
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import cv2
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## Carga del Modelo DINOv2

Usamos `facebook/dinov2-base` que es la version de tamaño medio. Hay versiones small, base, large y giant segun necesites mas precision o velocidad.

In [ ]:
model_name = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

print(f"Modelo DINOv2 cargado: {model_name}")
print(f"Dimension de embeddings: {model.config.hidden_size}")

## Carga de Imagenes de Ejemplo

In [ ]:
# Cargar imagenes desde GitHub
url_cars = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/cars.jpg"
image_cars = Image.open(BytesIO(requests.get(url_cars).content)).convert("RGB")

url_fruits = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/fruits.jpg"
image_fruits = Image.open(BytesIO(requests.get(url_fruits).content)).convert("RGB")

print("Imagenes cargadas")

## Extraer Embeddings de una Imagen

DINOv2 genera un embedding (vector numerico) que representa el contenido de la imagen. Imagenes similares tendran embeddings similares.

El modelo divide la imagen en patches (trozos) y genera un embedding para cada uno, mas un token especial `[CLS]` que representa la imagen completa.

In [ ]:
def extract_embeddings(image):
    """Extrae embeddings de imagen usando DINOv2"""
    
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS token: representa la imagen completa
    cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    # Patch tokens: cada region de la imagen
    patch_embeddings = outputs.last_hidden_state[:, 1:, :].cpu().numpy()
    
    print(f"Embedding global (CLS): {cls_embedding.shape}")
    print(f"Patch embeddings: {patch_embeddings.shape}")
    
    return cls_embedding, patch_embeddings

# Extraer embeddings de la imagen de coches
cls_emb, patch_emb = extract_embeddings(image_cars)

# Visualizar
plt.imshow(image_cars)
plt.title(f"Imagen dividida en {patch_emb.shape[1]} patches")
plt.axis("off")
plt.show()

## Similitud entre Imagenes

Podemos calcular la similitud entre dos imagenes usando sus embeddings. La similitud coseno nos dice que tan parecidas son (1 = identicas, 0 = no relacionadas).

In [ ]:
def cosine_similarity(emb1, emb2):
    """Calcula similitud coseno entre dos embeddings"""
    emb1 = emb1.flatten()
    emb2 = emb2.flatten()
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

# Comparar imagenes
cls_cars, _ = extract_embeddings(image_cars)
cls_fruits, _ = extract_embeddings(image_fruits)

similarity = cosine_similarity(cls_cars, cls_fruits)

print(f"\nSimilitud entre coches y frutas: {similarity:.4f}")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_cars)
axes[0].set_title("Coches")
axes[0].axis("off")
axes[1].imshow(image_fruits)
axes[1].set_title("Frutas")
axes[1].axis("off")
plt.suptitle(f"Similitud: {similarity:.4f}")
plt.tight_layout()
plt.show()

## Visualizacion de Patches con t-SNE

t-SNE es una tecnica para visualizar datos de alta dimension (como los embeddings de 768 dimensiones) en 2D. Patches similares aparecen cerca en el grafico.

Esto nos ayuda a entender que regiones de la imagen son similares entre si.

In [ ]:
def visualize_patches_tsne(image):
    """Visualiza patches de imagen usando t-SNE"""
    
    _, patch_emb = extract_embeddings(image)
    patch_emb = patch_emb[0]  # Quitar dimension de batch
    
    # Reducir dimension con PCA primero (mas rapido)
    pca = PCA(n_components=50)
    patch_emb_pca = pca.fit_transform(patch_emb)
    
    # Aplicar t-SNE para proyectar a 2D
    print("Calculando t-SNE... (puede tardar un poco)")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    patch_emb_2d = tsne.fit_transform(patch_emb_pca)
    
    # Calcular grid de patches (DINOv2 usa patches de 14x14)
    num_patches = patch_emb.shape[0]
    grid_size = int(np.sqrt(num_patches))
    
    # Visualizar
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Imagen original
    axes[0].imshow(image)
    axes[0].set_title("Imagen Original")
    axes[0].axis("off")
    
    # t-SNE projection
    scatter = axes[1].scatter(patch_emb_2d[:, 0], patch_emb_2d[:, 1], 
                            c=range(num_patches), cmap='viridis', s=50)
    axes[1].set_title(f"t-SNE de {num_patches} patches")
    axes[1].set_xlabel("t-SNE dimension 1")
    axes[1].set_ylabel("t-SNE dimension 2")
    plt.colorbar(scatter, ax=axes[1], label="Patch ID")
    
    plt.tight_layout()
    plt.show()
    
    return patch_emb_2d

# Visualizar patches de coches
tsne_result = visualize_patches_tsne(image_cars)

## Mapa de Similitud entre Regiones

Podemos seleccionar un patch especifico y ver que otros patches son similares a el. Esto es util para encontrar regiones relacionadas en la imagen.

In [ ]:
def similarity_map(image, reference_patch_idx=None):
    """Muestra mapa de similitud respecto a un patch de referencia"""
    
    _, patch_emb = extract_embeddings(image)
    patch_emb = patch_emb[0]  # Quitar dimension de batch
    
    num_patches = patch_emb.shape[0]
    grid_size = int(np.sqrt(num_patches))
    
    # Patch de referencia (por defecto el centro)
    if reference_patch_idx is None:
        reference_patch_idx = num_patches // 2
    
    ref_patch = patch_emb[reference_patch_idx]
    
    # Calcular similitud con todos los patches
    similarities = []
    for i in range(num_patches):
        sim = cosine_similarity(ref_patch, patch_emb[i])
        similarities.append(sim)
    
    similarities = np.array(similarities)
    
    # Reshape a grid
    sim_grid = similarities.reshape(grid_size, grid_size)
    
    # Visualizar
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Imagen Original")
    axes[0].axis("off")
    
    im = axes[1].imshow(sim_grid, cmap='hot', interpolation='nearest')
    axes[1].set_title(f"Similitud con patch {reference_patch_idx}")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], label="Similitud")
    
    plt.tight_layout()
    plt.show()
    
    return sim_grid

# Calcular mapa de similitud
sim_map = similarity_map(image_cars)

## Aplicacion: Encontrar Regiones Similares

Podemos usar DINOv2 para encontrar regiones similares en imagenes industriales, por ejemplo:
- Encontrar defectos similares en piezas
- Agrupar componentes parecidos
- Buscar patrones repetidos

In [ ]:
def find_similar_regions(image, top_k=5):
    """Encuentra las k regiones mas similares en la imagen"""
    
    _, patch_emb = extract_embeddings(image)
    patch_emb = patch_emb[0]
    
    num_patches = patch_emb.shape[0]
    grid_size = int(np.sqrt(num_patches))
    
    # Calcular similitud promedio de cada patch con todos los demas
    avg_similarities = []
    for i in range(num_patches):
        sims = [cosine_similarity(patch_emb[i], patch_emb[j]) for j in range(num_patches) if i != j]
        avg_similarities.append(np.mean(sims))
    
    # Encontrar patches mas representativos
    top_indices = np.argsort(avg_similarities)[-top_k:]
    
    print(f"Top {top_k} patches mas representativos:")
    for idx in top_indices:
        print(f"  Patch {idx}: similitud promedio = {avg_similarities[idx]:.4f}")
    
    # Visualizar patches en grid
    highlight_grid = np.zeros((grid_size, grid_size))
    for idx in top_indices:
        row = idx // grid_size
        col = idx % grid_size
        highlight_grid[row, col] = 1
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Imagen Original")
    axes[0].axis("off")
    
    axes[1].imshow(highlight_grid, cmap='Reds', interpolation='nearest')
    axes[1].set_title("Regiones mas representativas")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

find_similar_regions(image_cars, top_k=10)

In [1]:
# !pip install python-dotenv huggingface_hub

# # colab
# import os
# os.environ["HF_TOKEN"] = input("Introduce tu token HF: ")

# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])


# En el código
from dotenv import load_dotenv
import os
from huggingface_hub import login

# load_dotenv()  # Lee el archivo .env
# login(token=os.getenv("HF_TOKEN"))
# print("Login successful!")


# from transformers import AutoImageProcessor, AutoModel
# from transformers.image_utils import load_image


# url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# image = load_image(url)

# pretrained_model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"
# processor = AutoImageProcessor.from_pretrained(pretrained_model_name, token=os.getenv("HF_TOKEN"))
# model = AutoModel.from_pretrained(
#     pretrained_model_name, 
#     device_map="auto",
#     token=os.getenv("HF_TOKEN")
# )

# inputs = processor(images=image, return_tensors="pt").to(model.device)
# with torch.inference_mode():
#     outputs = model(**inputs)

# pooled_output = outputs.pooler_output
# print("Pooled output shape:", pooled_output.shape)


## Ejercicio: Compara dos imagenes diferentes

Usa las funciones que hemos creado para:
1. Extraer embeddings de `image_cars` e `image_fruits`
2. Calcular la similitud entre ellas
3. Visualizar los mapas de similitud de cada una

In [ ]:
# EJERCICIO: Tu codigo aqui

# Paso 1: Extraer embeddings
# ...

# Paso 2: Calcular similitud
# ...

# Paso 3: Visualizar mapas
# ...


## Ejercicio Extra (Comodin): Clustering de Patches

Usa K-Means para agrupar los patches en clusters y visualiza que partes de la imagen pertenecen a cada grupo.

**Pista**: Usa `sklearn.cluster.KMeans` sobre los embeddings de patches.

In [ ]:
# EJERCICIO EXTRA: Clustering de patches
from sklearn.cluster import KMeans

def cluster_patches(image, n_clusters=5):
    """Agrupa patches en clusters usando K-Means"""
    # Tu codigo aqui...
    pass

# Prueba:
# cluster_patches(image_cars, n_clusters=5)


## Resumen

En este notebook hemos aprendido:

✅ Que es el aprendizaje auto-supervisado  
✅ Como funciona DINOv2 como extractor de features  
✅ Extraer embeddings globales y por patches  
✅ Calcular similitud entre imagenes y regiones  
✅ Visualizar embeddings con t-SNE  
✅ Crear mapas de similitud  
✅ Aplicaciones en vision industrial  

**Siguiente paso**: En el proximo notebook veremos SAM2 para segmentacion universal y como combinarlo con Grounding DINO y Qwen2.5-VL.

**Contacto**: Si tienes dudas puedes escribirme un email!